In [ ]:
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitter import RecursiveCharacterTextSplitter
from langchain_core.embeddings import HuggingFaceEmbeddings
from langchain_chrome import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
#1. load a PDF document
loader = PyPDFLoader("python programming.pdf")
documents = loader.load()

In [ ]:
#2.Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100)

docs = text_splitter.split_documents(documents)

In [ ]:
#3. Create embeddings 
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
#4.store the embeddings in a vector database
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="RAG-collection"
)

In [ ]:
#5. Create a retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "lambda_mult": 0.5})

In [ ]:
#6.Initialize the LLM
llm = ChatOllama(model= "gpt-oss:120b-cloud")

In [ ]:
#7. Create a prompt template
prompt = PromptTemplate(
    template = """You are an AI assistant. Use the following context to answer the question.If the context or answer is not present in the context, say "I don't know".\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:""",

    input_variables = ["context", "question"]
)

In [ ]:
#8.Outputparser
parser = StrOutputParser()

#9build a rag chain
def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

rag_chain = (
    {
        "context":retriever|format_docs,
        "question":lambda x:x,
    }|prompt|llm|parser
)

In [ ]:
#10ask a question
query = "What is Python programming?"
result = rag_chain.invoke(query)

print(result)